# 28 — Rapid Fusion v2 with New MLS + Fracture

## Goal

No training. Rapidly combine the strongest available DEV components while Notebooks 22/25/26/27 continue running.

Inputs:
- Notebook 18 saved output: 2.5D ICH segmentation + slice predictions
- Notebook 23 saved output: new 2.5D fracture classifier
- Notebook 24 saved output: new 2.5D ordinal MLS model
- final-evaluation saved output: `common_dev_predictions.csv`

Key idea:
- keep the strong 2.5D ICH model,
- search a small set of continuity profiles,
- use the new ordinal MLS scores directly for the triage-relevant `>=3 mm` and `>=5 mm` decisions,
- use the new fracture classifier and calibrate only its decision threshold.

The locked TEST split is not used for tuning.

## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 2. Paths and auto-discovery

In [ ]:
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
OUTPUT_ROOT = Path("/kaggle/working/rapid_fusion_v2")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ICH_CLASSES = ["EDH", "SDH", "IPH", "SAH", "IVH"]
ICH_COLUMNS = [f"V_{name}" for name in ICH_CLASSES]

def unique_paths(paths):
    result, seen = [], set()
    for path in paths:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key); result.append(path)
    return result

def find_all(filename):
    matches = []
    for root in SEARCH_ROOTS:
        if root.exists():
            matches.extend(root.rglob(filename))
    return unique_paths(matches)

def choose_file(filename, required_text=None):
    matches = find_all(filename)
    if required_text is None:
        if len(matches) == 1:
            return matches[0]
        preferred = [p for p in matches if "final_evaluation" in str(p)]
        if len(preferred) == 1:
            return preferred[0]
    else:
        selected = [p for p in matches if required_text.lower() in str(p).lower()]
        if len(selected) == 1:
            return selected[0]

    print(f"Candidates for {filename}:")
    for path in matches:
        print(" -", path)
    raise FileNotFoundError(f"Could not uniquely identify {filename}.")

# Identify Notebook 18 root using 00_DIRECT_ANSWERS.csv content.
ich18_root = None
for path in find_all("00_DIRECT_ANSWERS.csv"):
    try:
        table = pd.read_csv(path)
    except Exception:
        continue
    text = " ".join(table.astype(str).fillna("").values.ravel()).lower()
    if "2.5d blood previous-center-next" in text and "expanded" in text:
        ich18_root = path.parent
        break

if ich18_root is None:
    raise FileNotFoundError("Attach the saved output of Notebook 18.")

ICH18_SLICE_PATHS = list(ich18_root.rglob("dev_slice_predictions.csv"))
if len(ICH18_SLICE_PATHS) != 1:
    raise FileNotFoundError("Could not uniquely find Notebook 18 dev_slice_predictions.csv.")
ICH18_SLICE_PATH = ICH18_SLICE_PATHS[0]

# Notebook 23
frac_root = None
for path in find_all("fracture_2p5d_config.json"):
    frac_root = path.parent.parent
    break
if frac_root is None:
    raise FileNotFoundError("Attach the saved output of Notebook 23.")
FRAC_SCORES_PATH = next(iter(frac_root.rglob("dev_slice_aggregates.csv")))

# Notebook 24
mls_root = None
for path in find_all("mls_2p5d_ordinal_config.json"):
    mls_root = path.parent.parent
    break
if mls_root is None:
    raise FileNotFoundError("Attach the saved output of Notebook 24.")
MLS_SCORES_PATH = next(iter(mls_root.rglob("dev_series_scores.csv")))
MLS_PREDS_PATH = next(iter(mls_root.rglob("dev_mls_predictions.csv")))

BASE_DEV_PATH = choose_file("common_dev_predictions.csv")

print("ICH18:", ICH18_SLICE_PATH)
print("Fracture23:", FRAC_SCORES_PATH)
print("MLS24:", MLS_SCORES_PATH)
print("Base DEV:", BASE_DEV_PATH)

## 3. Load DEV tables

In [ ]:
def norm_id(v):
    try: return str(int(float(v)))
    except Exception: return str(v).strip()

ich_slices = pd.read_csv(ICH18_SLICE_PATH)
frac = pd.read_csv(FRAC_SCORES_PATH)
mls = pd.read_csv(MLS_SCORES_PATH)
mls_direct = pd.read_csv(MLS_PREDS_PATH)
base = pd.read_csv(BASE_DEV_PATH)

for df in [ich_slices, frac, mls, mls_direct, base]:
    df["series_id"] = df["series_id"].map(norm_id)

ids = set(base.series_id) & set(frac.series_id) & set(mls.series_id) & set(ich_slices.series_id)
if len(ids) != 54:
    raise RuntimeError(f"Expected 54 common DEV series, found {len(ids)}.")

base = base[base.series_id.isin(ids)].copy().sort_values("series_id").reset_index(drop=True)
frac = frac[frac.series_id.isin(ids)].copy().sort_values("series_id").reset_index(drop=True)
mls = mls[mls.series_id.isin(ids)].copy().sort_values("series_id").reset_index(drop=True)
mls_direct = mls_direct[mls_direct.series_id.isin(ids)].copy().sort_values("series_id").reset_index(drop=True)

print("DEV series:", len(ids))

## 4. Official triage rule

In [ ]:
def triage_one(V_EDH, V_SDH, V_IPH, V_SAH, V_IVH, fracture_prob, MLS_mm):
    V_EDH=max(0.,float(V_EDH)); V_SDH=max(0.,float(V_SDH)); V_IPH=max(0.,float(V_IPH))
    V_SAH=max(0.,float(V_SAH)); V_IVH=max(0.,float(V_IVH)); MLS_mm=max(0.,float(MLS_mm))
    total=V_EDH+V_SDH+V_IPH+V_SAH+V_IVH
    has_ich=total>=0.1; fracture_present=float(fracture_prob)>=0.5

    if MLS_mm>=5 and (has_ich or fracture_present): return 2
    if V_EDH>=30: return 2
    if V_SDH>=70: return 2
    if V_IPH>=70: return 2
    if total>=60: return 2
    if has_ich and MLS_mm>=3 and total>=40: return 2
    if fracture_present and total>=15: return 2
    if MLS_mm>=5 and not (has_ich or fracture_present): return 1
    if has_ich: return 1
    if 3<=MLS_mm<5: return 1
    if fracture_present and total<15: return 1
    if total>=0.1 and MLS_mm>=1: return 1
    return 0

def evaluate(ich_df, mls_values, fracture_values):
    work = base[["series_id","true_triage"]].merge(ich_df,on="series_id",how="inner").sort_values("series_id").reset_index(drop=True)
    mls_values=np.asarray(mls_values,float); fracture_values=np.asarray(fracture_values,float)
    pred=[]
    for i,row in enumerate(work.itertuples(index=False)):
        pred.append(triage_one(row.V_EDH,row.V_SDH,row.V_IPH,row.V_SAH,row.V_IVH,fracture_values[i],mls_values[i]))
    pred=np.asarray(pred)
    return float(f1_score(work.true_triage,pred,average="macro",labels=[0,1,2],zero_division=0)), float(accuracy_score(work.true_triage,pred)), pred

## 5. Build ICH continuity candidates from Notebook 18

In [ ]:
ICH_PROFILES = {
    "raw": {"EDH":1,"SDH":1,"IPH":1,"SAH":1,"IVH":1},
    "all2": {"EDH":2,"SDH":2,"IPH":2,"SAH":2,"IVH":2},
    "all3": {"EDH":3,"SDH":3,"IPH":3,"SAH":3,"IVH":3},
    "sah1_others2": {"EDH":2,"SDH":2,"IPH":2,"SAH":1,"IVH":2},
    "edh2_sdh2_iph2_sah1_ivh3": {"EDH":2,"SDH":2,"IPH":2,"SAH":1,"IVH":3},
}

def filtered_volume(group, subtype, min_run):
    g=group.sort_values("slice_order")
    flags=g[f"pred_positive_{subtype}"].astype(bool).to_numpy()
    vols=g[f"pred_volume_{subtype}"].to_numpy(float)
    keep=np.zeros(len(g),bool); start=None
    for i in range(len(flags)+1):
        active=i<len(flags) and flags[i]
        if active and start is None: start=i
        if not active and start is not None:
            if i-start>=min_run: keep[start:i]=True
            start=None
    return float(vols[keep].sum())

def aggregate_ich(profile):
    rows=[]
    for sid,g in ich_slices.groupby("series_id",sort=False):
        row={"series_id":sid}
        for subtype in ICH_CLASSES:
            row[f"V_{subtype}"]=filtered_volume(g,subtype,profile[subtype])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("series_id").reset_index(drop=True)

ich_candidates={name:aggregate_ich(profile) for name,profile in ICH_PROFILES.items()}
print("ICH profiles:", list(ich_candidates))

## 6. Important MLS simplification

For the official triage rule, `1 mm` is not a meaningful decision boundary for a model that already outputs ICH presence through volume.

The triage-changing MLS boundaries are primarily:
- `>= 3 mm`
- `>= 5 mm`

So this notebook searches the new ordinal model's `p3` and `p5` scores directly.

## 7. Stage A — choose ICH profile with current non-ICH models

In [ ]:
current_mls = base["pred_MLS_mm"].to_numpy(float)
current_frac = base["pred_fracture_prob"].to_numpy(float)

rows=[]
for name,ich_df in ich_candidates.items():
    f1,acc,_=evaluate(ich_df,current_mls,current_frac)
    rows.append({"ICH_profile":name,"macro_F1":f1,"accuracy":acc})
ich_search=pd.DataFrame(rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
display(ich_search)
best_ich_name=str(ich_search.iloc[0].ICH_profile)
best_ich=ich_candidates[best_ich_name]

## 8. Stage B — search ordinal MLS >=3 and >=5 decisions

In [ ]:
MLS_AGGS = ["max","top3","top5","top10"]
THRESHOLDS = np.arange(0.05,0.96,0.02)

mls_rows=[]
for agg3 in MLS_AGGS:
    p3=mls[f"p3_{agg3}"].to_numpy(float)
    for th3 in THRESHOLDS:
        flag3=p3>=th3
        for agg5 in MLS_AGGS:
            p5=mls[f"p5_{agg5}"].to_numpy(float)
            for th5 in THRESHOLDS:
                flag5=p5>=th5
                values=np.zeros(len(mls),float)
                values[flag3]=3.5
                values[flag5]=5.5
                f1,acc,_=evaluate(best_ich,values,current_frac)
                mls_rows.append({"agg3":agg3,"th3":float(th3),"agg5":agg5,"th5":float(th5),"macro_F1":f1,"accuracy":acc})

mls_search=pd.DataFrame(mls_rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
mls_search.to_csv(OUTPUT_ROOT/"01_mls_triage_search.csv",index=False)
display(mls_search.head(20))

best_mls=mls_search.iloc[0]
flag3=mls[f"p3_{best_mls.agg3}"].to_numpy(float)>=float(best_mls.th3)
flag5=mls[f"p5_{best_mls.agg5}"].to_numpy(float)>=float(best_mls.th5)
best_mls_values=np.zeros(len(mls),float); best_mls_values[flag3]=3.5; best_mls_values[flag5]=5.5

## 9. Stage C — new fracture classifier calibration

In [ ]:
FRAC_AGGS=["max","top3_mean","top5_mean","top10_mean"]
FRAC_THRESHOLDS=np.arange(0.05,0.96,0.01)

frac_rows=[]
for agg in FRAC_AGGS:
    scores=frac[agg].to_numpy(float)
    for th in FRAC_THRESHOLDS:
        values=(scores>=th).astype(float)
        f1,acc,_=evaluate(best_ich,best_mls_values,values)
        frac_rows.append({"aggregator":agg,"threshold":float(th),"macro_F1":f1,"accuracy":acc})

frac_search=pd.DataFrame(frac_rows).sort_values(["macro_F1","accuracy"],ascending=False).reset_index(drop=True)
frac_search.to_csv(OUTPUT_ROOT/"02_fracture_triage_search.csv",index=False)
display(frac_search.head(20))

best_frac=frac_search.iloc[0]
best_frac_values=(frac[best_frac.aggregator].to_numpy(float)>=float(best_frac.threshold)).astype(float)

## 10. Final comparison

In [ ]:
baseline_f1,baseline_acc,_=evaluate(ich_candidates["raw"],current_mls,current_frac)
ich_f1,ich_acc,_=evaluate(best_ich,current_mls,current_frac)
new_mls_f1,new_mls_acc,_=evaluate(best_ich,best_mls_values,current_frac)
final_f1,final_acc,final_pred=evaluate(best_ich,best_mls_values,best_frac_values)

direct_mls = mls_direct["pred_MLS_mm"].to_numpy(float)
direct_mls_f1,direct_mls_acc,_=evaluate(best_ich,direct_mls,current_frac)

summary=pd.DataFrame([
    {"configuration":"2.5D raw ICH + current MLS + current fracture","macro_F1":baseline_f1,"accuracy":baseline_acc},
    {"configuration":f"ICH {best_ich_name} + current MLS + current fracture","macro_F1":ich_f1,"accuracy":ich_acc},
    {"configuration":f"ICH {best_ich_name} + Notebook24 direct MLS bins + current fracture","macro_F1":direct_mls_f1,"accuracy":direct_mls_acc},
    {"configuration":f"ICH {best_ich_name} + optimized Notebook24 ordinal MLS + current fracture","macro_F1":new_mls_f1,"accuracy":new_mls_acc},
    {"configuration":"Final: ICH + ordinal MLS + new fracture","macro_F1":final_f1,"accuracy":final_acc},
])
summary.to_csv(OUTPUT_ROOT/"00_DIRECT_ANSWERS.csv",index=False)
display(summary)

print("Best ICH profile:",best_ich_name)
print("Best MLS:",dict(best_mls))
print("Best fracture:",dict(best_frac))

## 11. Save deployable fusion config

In [ ]:
config={
    "ICH_profile":best_ich_name,
    "ICH_profile_rules":ICH_PROFILES[best_ich_name],
    "MLS":{
        "agg3":str(best_mls.agg3),
        "threshold3":float(best_mls.th3),
        "agg5":str(best_mls.agg5),
        "threshold5":float(best_mls.th5),
        "outputs_mm":{"none":0.0,"ge3":3.5,"ge5":5.5},
    },
    "fracture":{
        "aggregator":str(best_frac.aggregator),
        "threshold":float(best_frac.threshold),
    },
    "DEV_macro_F1":float(final_f1),
    "DEV_accuracy":float(final_acc),
    "locked_test_used_for_tuning":False,
}
with open(OUTPUT_ROOT/"fusion_v2_config.json","w",encoding="utf-8") as f:
    json.dump(config,f,indent=2)
print(json.dumps(config,indent=2))

# Next

When Notebooks 22, 25, 26, and 27 finish, compare them against this frozen DEV pipeline.

If Notebook 26 provides a high-recall ICH gate, test it before changing the segmentation model again. If Notebook 27 improves the MLS `>=3 mm` weakness, replace Notebook 24 in this same fusion search.